# Drain parser for BGL (Blue Gene/L) logs

Mirror of `Parser.ipynb` adapted to the LogHub **BGL** dataset.

**Format recap** (whitespace-separated, 9 fixed header fields):

```text
<LABEL> <UNIX_TS> <DATE> <NODE> <DATETIME> <NODE> <COMPONENT> <SUBCOMPONENT> <LEVEL> <MESSAGE>
```

`<LABEL>` is `-` for normal events and a short alert code (e.g. `KERNDTLB`, `APPSEV`) for anomalies.
Unlike HDFS, **anomaly labels are inline** — no separate CSV is needed.

In [ ]:
import os, sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))  # project root

# ── Run tag ────────────────────────────────────────────────────
# Auto-generated as YYYYMMDD_HHMM so multiple daily runs stay distinct.
# Override before running to reload or continue a specific experiment:
#   RUN_TAG = "20260421_0800"
RUN_TAG  = datetime.now().strftime("%Y%m%d_%H%M")
RUN_DATE = RUN_TAG.split("_")[0]

# Dataset family prefix keeps BGL artefacts distinct from HDFS ones under data/processed/.
DATASET = "bgl"
ARTIFACT_PREFIX = f"{RUN_TAG}_{DATASET}"

print(f"Run tag        : {RUN_TAG}")
print(f"Dataset prefix : {DATASET}")
print(f"Outputs will be prefixed with: {ARTIFACT_PREFIX}_")


In [ ]:
from src.parser import BGLParser

config_path  = "../configs/drain_bgl.ini"
bgl_log_path = "../data/raw/BGL_full.log"

n_lines = sum(1 for _ in open(bgl_log_path, "rb"))
print(f"Total lines in log: {n_lines:,}")

parser = BGLParser(config_path=config_path)

# First pass: learn templates
parser.fit_file(bgl_log_path)

# Second pass: annotate every line with its final cluster_id, template, params,
# label, node_id, component/subcomponent/level.
df = parser.annotate_file(bgl_log_path, max_lines=None)
print(df.shape)
df.head(3)

parser.export_templates(f"../data/processed/{ARTIFACT_PREFIX}_templates.json")
parser.save(f"../models/{ARTIFACT_PREFIX}_drain_parser.bin")


## Anomaly-rate sanity check

Per LogHub documentation, BGL has **~7.2 %** anomalous lines (332 222 of 4 631 261).

In [ ]:
n_total   = len(df)
n_anomaly = int(df["is_anomaly"].sum())
n_normal  = n_total - n_anomaly

print(f"Total lines    : {n_total:,}")
print(f"Normal (-)     : {n_normal:,}  ({n_normal / n_total:.2%})")
print(f"Anomalous      : {n_anomaly:,}  ({n_anomaly / n_total:.2%})")
print()
print("Top 10 alert codes (excluding '-'):")
print(df.loc[df["is_anomaly"], "label"].value_counts().head(10))
print()
print("Log-level distribution:")
print(df["level"].value_counts())


## Persistence

Same fastparquet workaround as `Parser.ipynb`:
  1. JSON-encode list-valued columns (`parameters`) before writing.
  2. Cast Arrow-backed string dtypes back to plain `object` so fastparquet can serialise.

In [ ]:
import json

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH  = PROCESSED_DIR / f"{ARTIFACT_PREFIX}_annotated.parquet"

df_save = df.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# fastparquet cannot write pandas Arrow-backed string columns (ArrowDtype).
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

df_save.to_parquet(PARQUET_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows  →  {PARQUET_PATH}")
print(f"File size : {PARQUET_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"Columns   : {df_save.columns.tolist()}")


In [ ]:
parser.validate()

## Next steps

1. **Enrich templates** — reuse `Enricher.enrich_corpus_bgl` (declared in `src/enricher/enricher.py`). The output goes to `data/processed/{ARTIFACT_PREFIX}_templates_enriched.json`.
2. **Sequencer** — BGL has no native `block_id`. The two canonical options are:
   * Group by `node_id` (one sequence per compute node).
   * Slide fixed time windows over `timestamp` (e.g. 6 h), standard in DeepLog / LogBERT.
3. **PrepareDataset / GAE_Training** — same pipeline as HDFS, with the single substitution `block_id → (node_id | window_id)` as the per-graph key.